# Reproducing Park et al. (2019) from CRCNS hc-20

**Paper.** Park EH, Keeley S, Savin C, Ranck JB Jr, Fenton AA (2019). How the internally
organized direction sense is used to navigate. *Neuron* 101:285-293.
[doi:10.1016/j.neuron.2018.11.019](https://doi.org/10.1016/j.neuron.2018.11.019). The claims
file (`claims/park-2019.yaml`) was written from the version of record, with its STAR Methods
and Supplemental Information.

**Dataset.** CRCNS hc-20 (Park and Fenton 2018), [doi:10.6080/K0SF2TDN](https://doi.org/10.6080/K0SF2TDN),
streamed from the public bucket (`hc-20/live/data.zip`, read in place with fsspec and zipfile).
CRCNS data sets are not versioned, so the version is recorded as `unversioned`; the files read
are those of `data.zip` with MD5 `b82c7b97b0ba3491f04ecc4dbcd48e83` in the data set's
`checksums.md5` (the next cell checks the bucket's ETag against it). This is the only published
version, and the one the paper cites (by DOI), so it is the one pinned. Its data descriptor is
version 0.6 (14 November 2018).

The dataset holds the 115 head-direction cells (one TSD file per cell and trial: stable1, the
rotating trial in room and in arena coordinates, stable2) from 13 rats in 44 sessions, and the
tracker files of each trial. It holds no other MEC cells, no LFP, no waveforms and no training
sessions.

| Claim | Tested here |
|---|---|
| cell-counts | tested (head-direction cells only) |
| hd-cell-rate | tested |
| fig1a-avoidance-learning | blocked: data missing (training sessions) |
| fig1c-speed | tested |
| fig1c-lfp-power | blocked: data missing (LFP) |
| fig1c-firing-rate | tested |
| fig1c-directionality | tested |
| fig1d-preference-change | tested |
| fig1d-instability | tested |
| fig2-decoding-stable-vs-rotating | tested |
| figs2-frame-preference | tested |
| fig2c-proximity-anticorrelation | tested |
| figs3-no-head-direction-bias | tested |
| fig3-hd-pair-correlations-preserved | tested |
| figs4-all-pair-correlations-preserved | blocked: data missing (non-head-direction cells) |
| figs5-hd-cells-high-model-information | blocked: data missing (the 921 modelled cells) |
| figs6-subclass-correlations-preserved | blocked: data missing (depends on figs5) |
| no-angular-velocity-tuning | tested |
| fig4-internal-tuning-preserved | tested |
| fig4c-arena-registration-short-lived | tested |
| fig5-error-inhomogeneous | tested |
| fig5c-error-minima-at-cues | tested |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import itertools, resource, time, warnings
import numpy as np
import pandas as pd
from scipy import stats
import hc20
warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
T0 = time.time()
D = hc20.load()
units, trk, spk = D["units"], D["trk"], D["spk"]
print(f"loaded {len(units)} units in {time.time() - T0:.0f} s")

In [ ]:
import urllib.request
req = urllib.request.Request(hc20.URL, method="HEAD")
with urllib.request.urlopen(req) as r_:
    etag = r_.headers["ETag"].strip('"')
print("data.zip ETag:", etag, "; matches checksums.md5:", etag == "b82c7b97b0ba3491f04ecc4dbcd48e83")

## Data check

The zip holds, under `data/TSD`, one file per unit in each of SA1 (stable1), RA_rm (rotating,
room frame), RA_ar (rotating, arena frame) and SA2 (stable2), and `FileNameList.xlsx`; under
`data/DAT`, the tracker files. The cell below lists the zip, counts units by rat and session,
checks the file list against the files, the sampling rates, the tracks shared by the units of a
session, the spike clock, duplicates, and the shock zone geometry in the tracker headers.

In [ ]:
z = D["z"]
names = [n for n in z.namelist() if not n.endswith("/")]
print("members:", len(names))
print(pd.Series([n.rsplit("/", 1)[0] for n in names]).value_counts().to_string())
print("\nunits per rat (dataset) against Table S1:")
table_s1 = {"R024": 5, "R027": 3, "R029": 41, "R030": 0, "R035": 17, "R039": 6, "R084": 2, "R130": 1,
            "R157": 25, "R176": 1, "R357": 8, "R521": 3, "R523": 1, "R848": 2, "R850": 0}
per_rat = units.groupby("rat").size()
cmp = pd.DataFrame({"dataset": per_rat, "Table S1": pd.Series(table_s1)}).fillna(0).astype(int)
print(cmp.to_string()); print("total", cmp.sum().to_dict())
sess_n = units.groupby("session").size()
print(f"\n{units.session.nunique()} sessions; units per session:", sess_n.value_counts().sort_index().to_dict())
meta = D["meta"]
print("\nsamples per file by trial:\n", meta.groupby("trial").n_samples.describe()[["min", "50%", "max"]].to_string())
print("header sampling rates (samples/s):", sorted(meta.rate.round(6).unique()))
print("files with an odd rate:\n", meta[meta.rate > 31][["session", "unit", "trial", "rate", "t0", "n_samples", "n_spikes"]].to_string())
print("trailing bytes after the last sample, max:", meta.trailing.max())

In [ ]:
# FileNameList.xlsx against the match by session and tetrode-cluster
fl = D["filelist"].iloc[1:].reset_index(drop=True)
fl.columns = ["S1", "RR", "RA", "S2"]
mism = []
for _, row in fl.iterrows():
    k1, k2 = hc20.unit_key(row.S1), hc20.unit_key(row.S2)
    kr, ka = hc20.unit_key(row.RR), hc20.unit_key(row.RA)
    if not (k1 == k2 == kr == ka):
        mism.append(row.to_dict())
print(f"FileNameList rows: {len(fl)}; rows whose four names are not the same session and tetrode-cluster: {len(mism)}")
print(pd.DataFrame(mism).to_string())
print("\nstable2 files not named SA2:", [f for f in units.file_S2 if "SA2" not in f])

In [ ]:
# tracks: units of a session share one track per trial; duplicates across sessions
h = meta.groupby(["session", "trial"]).track_hash.nunique()
print("session-trials whose units hold different tracks:", h[h > 1].to_dict())
print("unit files with their own track:", [k for k in trk if len(k) == 3],
      {k: len(v["d"]) for k, v in trk.items() if len(k) == 3})
first = meta.drop_duplicates(["session", "trial"])
dup = first[first.duplicated("track_hash", keep=False)]
print("identical tracks in different sessions:\n", dup[["session", "trial", "t0", "n_samples"]].to_string())

def radius(t):
    r = np.hypot(t["x"] - 127.5, t["y"] - 127.5).astype(float)
    r[(t["x"] == 0) & (t["y"] == 0)] = np.nan
    return r
rad_dev = {}
for s in units.session.unique():
    a, b = radius(trk[(s, "RR")]), radius(trk[(s, "RA")])
    n = min(len(a), len(b))
    rad_dev[s] = np.nanmedian(np.abs(a[:n] - b[:n]))
rad_dev = pd.Series(rad_dev)
print("\nrotating trial, median |distance from centre, room track - arena track| (px): median over sessions "
      f"{rad_dev.median():.2f}, max {rad_dev.max():.2f} in {rad_dev.idxmax()}; sessions above 2 px: {rad_dev[rad_dev > 2].round(2).to_dict()}")

# spike trains: rotating room and arena files hold the same spikes; duplicates across trials or units
same_rr_ra = all(np.array_equal(v["RR"]["t"], v["RA"]["t"]) for v in spk.values())
print("RA_rm and RA_ar spike times identical for every unit:", same_rr_ra)
seen = {}
for k, v in spk.items():
    for tr in ("S1", "RR", "S2"):
        seen.setdefault(v[tr]["t"].tobytes(), []).append((k, tr))
print("identical spike trains in different files:", [v for v in seen.values() if len(v) > 1])

In [ ]:
# unit-trials whose files are shown wrong above; left out in the "clean" variant of each analysis
R029_WARP = ["1009R029", "1015R029", "1023R029", "1025R029"]
EXCL = {(("1024R157", "P9C1"), "RR"), (("1024R157", "P9C1"), "RA"), (("1121R157", "P11C2"), "S2")}
for s_, u_ in units[["session", "unit"]].itertuples(index=False):
    if s_ == "1211R039":
        EXCL.add(((s_, u_), "RR"))
    if s_ in R029_WARP:
        EXCL |= {((s_, u_), "RR"), ((s_, u_), "RA")}
print(f"unit-trials left out in the clean variant: {len(EXCL)}, units affected: {len({k for k, _ in EXCL})}")

Problems in the files, each shown by the cells above:

- Unit 1024R157 P9C1: its rotating-trial spike times are identical to its stable1 spike times
  (same count, same times), while the tracks differ, so one of the two files holds the wrong
  spikes.
- Session 1211R039: its rotating room-frame track is identical to that of 1212R039 and does
  not agree with its own arena-frame track (the distance from the arena centre, which rotation
  does not change, differs by a median of 22 px, against 0.3 px in other sessions). The
  .dat files of 1211R039 agree with each other, and its arena-frame TSD track matches its
  arena .dat file exactly.
- Sessions 1009R029, 1015R029, 1023R029 and 1025R029: the room-frame and arena-frame TSD
  tracks of the rotating trial disagree (median 7 to 13 px at the best constant lag) and
  neither matches the session's .dat files exactly; the .dat files agree with each other.
  The lag between the TSD tracks and the .dat track grows through the trial (shown in the
  cell below), so the two TSD frames were built with different timing and spikes cannot be
  aligned to both.
- Unit 1121R157 P11C2: its stable2 file (`cl1`) holds a track of half the length of the
  other units' (26954 samples).
- Unit 1017R035 P11C2: its stable2 header states 2000 samples/s; its spike times span about
  30 min over 54185 samples, so the rate of the other files is used.

The outcome of each claim is decided on the files as deposited, as the paper's code reads them:
the paper's counts (115 cells, seven ensembles of at least five cells, about 142 pairs) imply it
used all of them, so this is the test of what the paper reports from these files. Each analysis
that uses a rotating or stable2 trial also shows a "clean" variant that leaves out the unit-trials
listed above (for the ensemble analyses, session 1023R029 in the rotating trial). Where the clean
variant gives another result, the section's conclusion says so, and the record's notes say that
the outcome depends on files shown wrong here.

In [ ]:
# spike clock against the tracker samples: each spike sits in a sample; compare its timestamp with
# the sample's time from the header (first timestamp + index / rate)
rows = []
for k in [tuple(x) for x in units[["session", "unit"]].values]:
    for tr in ("S1", "RR", "S2"):
        t = hc20.track(D, k, tr)
        s = spk[k][tr]
        if len(s["i"]) < 20:
            continue
        off = s["t"] * t["rate"] - s["i"]  # samples
        a, b = np.polyfit(s["i"], off, 1)
        res = off - (a * s["i"] + b)
        rows.append(dict(session=k[0], unit=k[1], trial=tr, drift_per_1e4=a * 1e4, offset_end=a * len(t["d"]) + b,
                         resid_min=res.min(), resid_max=res.max()))
clk = pd.DataFrame(rows)
print(clk.describe().round(3).to_string())
odd = clk[(clk.resid_min < -3) | (clk.resid_max > 3)]
print(f"files with a spike more than 3 samples off the line: {len(odd)} of {len(clk)}")
print(odd.round(1).to_string(index=False))

Each spike is stored inside a tracker sample. Its timestamp runs ahead of the sample's time
from the header (first timestamp plus index over rate) by a drift that grows linearly through
the trial (about 0.5 samples per 1000, reaching about 25 samples, 0.8 s, at 30 min). In most
files the residuals about the line stay within 1.5 samples; 21 files have a few spikes further
off, and the stable2 file of 1017R035 P11C2 (the one with a 2000 samples/s header) many. So the spikes and the samples share one
clock up to a rate that differs from the header's by about 0.05%. Analyses here place spikes
by the sample that holds them, as the paper's decoder does, and use the spike timestamps only
for intervals between spikes (pair correlations, internally referenced tuning). The check
that tuning depends on the alignment of spikes and directions is below (shifted directions).

In [ ]:
# arena, units and the shock zones from the tracker headers of the rotating trials
dat_rows, dat_tabs = [], {}
for n in sorted(names):
    if not n.startswith("data/DAT/RA/") or not n.endswith(".dat"):
        continue
    f = n.split("/")[-1]
    hdr, tab = hc20.read_dat(z.read(n))
    fr = "Arena" if "ArenaTrack" in f else "Room"
    sec = [float(v) for v in hc20.dat_value(hdr, "ReinforcedSector")]
    dat_rows.append(dict(session=hc20.dat_session(f), frame=fr, center=sec[0], width=sec[1], r_in=sec[2], r_out=sec[3],
                         pix_per_cm=float(hc20.dat_value(hdr, "TrackerResolution_PixPerCM")[0]),
                         diameter_m=float(hc20.dat_value(hdr, "ArenaDiameter_m")[0]),
                         centre=" ".join(hc20.dat_value(hdr, "ArenaCenterXY")), n=len(tab)))
    dat_tabs[(hc20.dat_session(f), fr)] = tab
DAT = pd.DataFrame(dat_rows)
print(f"rotating .dat files: {len(DAT)} for {DAT.session.nunique()} sessions")
print(DAT.groupby(["width", "r_in", "r_out", "center"]).size().to_string())
print("pixels per cm:", DAT.pix_per_cm.unique(), "arena diameter (m):", DAT.diameter_m.unique(), "centre:", DAT.centre.unique())
# the angle convention of the sector: mean angle of the samples the tracker flags as inside it
conv = []
for (s, fr), t in dat_tabs.items():
    bit = 1 if fr == "Room" else 2
    m = (t.Sectors.fillna(0).astype(int) & bit) > 0
    if m.sum() < 5:
        continue
    x, y = t.RoomX[m] - 127.5, t.RoomY[m] - 127.5
    conv.append(dict(session=s, frame=fr, n=int(m.sum()),
                     y_down=np.degrees(np.angle(np.mean(np.exp(1j * np.arctan2(-y, x).to_numpy())))) % 360,
                     y_up=np.degrees(np.angle(np.mean(np.exp(1j * np.arctan2(y, x).to_numpy())))) % 360))
conv = pd.DataFrame(conv)
print("mean angle of in-sector samples (sector centre 90 in the header):\n", conv[["y_down", "y_up"]].describe().round(1).to_string())
# the TSD tracks against the .dat files: positions of one session
s = "1022R029"
tab = dat_tabs[(s, "Room")]
t = trk[(s, "RR")]
okd = (t["x"] > 0)
print(f"{s}: first tracked TSD sample x,y,dir =", t["x"][okd][0], t["y"][okd][0], t["d"][okd][0],
      "; first .dat sample RoomX,RoomY,Angle =", tab.RoomX.iloc[0], tab.RoomY.iloc[0], tab.Angle.iloc[0])

In [ ]:
# the lag of the TSD rotating tracks against the tracker file, in 3000-sample segments
def rad_xy(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    r = np.hypot(x - 127.5, y - 127.5); r[(x == 0) & (y == 0)] = np.nan
    return r
for s_ in R029_WARP + ["1022R029"]:
    dr = rad_xy(dat_tabs[(s_, "Room")].RoomX, dat_tabs[(s_, "Room")].RoomY)
    out = {}
    for tr in ("RR", "RA"):
        a = rad_xy(trk[(s_, tr)]["x"], trk[(s_, tr)]["y"])
        lags = []
        for w0 in (6000, 24000, 48000):
            seg = a[w0:w0 + 3000]
            c = [(np.nanmedian(np.abs(seg - dr[w0 + L:w0 + L + 3000])), L) for L in range(-1500, 1500, 3)
                 if 0 <= w0 + L and w0 + L + 3000 <= len(dr)]
            lags.append(min(c)[1])
        out[tr] = lags
    print(s_, "lag (samples) of the TSD track behind the room .dat track at samples 6000, 24000, 48000:", out)

## Per-unit tuning

Tuning curves at 10 degree resolution, rate = spikes over time in each bin (STAR Methods, Data
Analysis), per unit in stable1 (S1), the rotating trial in room (RR) and arena (RA) frames,
stable2 (S2), and the first and second 15 min of stable1 (H1, H2; the claims file's reading
of "the two halves"). Samples with direction above 360 are dropped, as the paper's code does.
Each unit uses the track stored in its own file.

In [ ]:
def unit_measures(key):
    out = {"session": key[0], "unit": key[1]}
    for tr in ("S1", "RR", "RA", "S2"):
        t = hc20.track(D, key, tr)
        si = spk[key][tr]["i"]
        n = len(t["d"])
        out[f"rate_{tr}"] = len(si) / (n / t["rate"])
        cur, cnt, occ = hc20.tuning(t["d"], si, t["rate"])
        out[f"curve_{tr}"], out[f"cnt_{tr}"] = cur, cnt
        out[f"dir_{tr}"] = hc20.resultant(cur)[0]
        out[f"pref_{tr}"] = hc20.pref_dir(cur)
        if tr == "S1":
            # head-direction criteria (STAR Methods, Data Analysis): Rayleigh test of the tuning
            # curve's vector (n = spikes with a valid direction; the variant n = 36 bins) and
            # Kuiper's test of the spike directions against the sampled directions
            nsp = int(cnt.sum())
            out["rayleigh_p"] = hc20.rayleigh_p(out["dir_S1"], nsp) if nsp else np.nan
            out["rayleigh_p_bins"] = hc20.rayleigh_p(out["dir_S1"], 36)
            sd = t["d"][si]
            sd = np.radians(sd[~np.isnan(sd)])
            alld = np.radians(t["d"][~np.isnan(t["d"])])
            out["kuiper_p"] = hc20.kuiper_two(sd, alld)[2] if len(sd) > 1 else np.nan
            # alignment check: the same spikes against directions shifted by 60 s
            dsh = np.roll(t["d"], int(60 * t["rate"]))
            out["dir_S1_shift60"] = hc20.resultant(hc20.tuning(dsh, si, t["rate"])[0])[0]
            half = np.arange(n) < n // 2
            for h, m in (("H1", half), ("H2", ~half)):
                c, k, _ = hc20.tuning(t["d"], si, t["rate"], mask=m)
                out[f"curve_{h}"], out[f"cnt_{h}"], out[f"pref_{h}"] = c, k, hc20.pref_dir(c)
                out[f"dir_{h}"] = hc20.resultant(c)[0]
    return out

UM = pd.DataFrame([unit_measures(tuple(k)) for k in units[["session", "unit"]].values])
UM["rat"] = UM.session.str[4:]
UM["key"] = list(zip(UM.session, UM.unit))
def clean(tr):
    return np.array([(k, tr) not in EXCL for k in UM.key])
print("rates (AP/s) by trial:\n", UM[[f"rate_{t}" for t in ("S1", "RR", "S2")]].describe().round(2).to_string())
print("\ndirectionality (Rayleigh vector length) by condition:\n", UM[[f"dir_{t}" for t in ("S1", "RR", "RA", "S2", "H1", "H2")]].describe().round(3).to_string())
print("\nhighest rates:", sorted([(round(r_, 1), k, t_) for k, row in UM.set_index("key").iterrows()
                                  for t_, r_ in (("S1", row.rate_S1), ("RR", row.rate_RR), ("S2", row.rate_S2))], reverse=True)[:3])
print("\nalignment check, stable1 directionality: true", UM.dir_S1.median().round(3), "; directions shifted by 60 s",
      UM.dir_S1_shift60.median().round(3), "; units higher with the true alignment:", int((UM.dir_S1 > UM.dir_S1_shift60).sum()), "of", len(UM))

## cell-counts

> The paper analyses 115 dorsal MEC head-direction cells from 13 rats, among 353 MEC cells
> forming 556 cell pairs (Table S1), out of 1594 MEC single units recorded.

Reading (a) counts the units and simultaneously recorded pairs the dataset holds; reading (b)
applies the paper's head-direction criteria in stable1 (mean rate below 10 AP/s, Rayleigh
test and Kuiper's test each at p < 0.05). The dataset holds only the head-direction cells (one
FileNameList row per unit, 115 rows), so the counts of 353 MEC cells and 556 MEC pairs cannot
be made.

In [ ]:
sess_n = units.groupby("session").size()
hd_pairs = int(sum(n * (n - 1) // 2 for n in sess_n))
print(f"(a) units: {len(units)} (expected 115 +- 11.5); rats: {units.rat.nunique()} (expected 13); "
      f"simultaneously recorded head-direction pairs: {hd_pairs} (paper: 142, Results, Figure 4)")
crit = (UM.rate_S1 < 10) & (UM.rayleigh_p < 0.05) & (UM.kuiper_p < 0.05)
crit_bins = (UM.rate_S1 < 10) & (UM.rayleigh_p_bins < 0.05) & (UM.kuiper_p < 0.05)
print(f"(b) units passing the head-direction criteria in stable1: {int(crit.sum())} of {len(UM)} "
      f"(Rayleigh n = spikes); with the Rayleigh n = 36 bins: {int(crit_bins.sum())}")
print("    failing: rate >= 10:", int((UM.rate_S1 >= 10).sum()), "; Rayleigh p >= 0.05:", int((UM.rayleigh_p >= 0.05).sum()),
      "(bins:", int((UM.rayleigh_p_bins >= 0.05).sum()), "); Kuiper p >= 0.05:", int((UM.kuiper_p >= 0.05).sum()))
pr = pd.DataFrame({"dataset (a)": per_rat, "pass (b)": UM[crit].groupby("rat").size(), "Table S1": pd.Series(table_s1)}).fillna(0).astype(int)
print(pr.to_string())
print("per-rat counts that differ from Table S1 under (a):", pr[pr["dataset (a)"] != pr["Table S1"]].index.tolist())
print("MEC cells (353) and MEC pairs (556): not countable, the dataset holds only the head-direction cells")
# Table S1 differences (R029 40 against 41, R523 2 against 1): the TSD headers hold no date or rat
# (cell 4 shows their fields), so the dates come from the tracker headers of the sessions of 1 October
print("\nunits of the sessions recorded on a 1 October:")
print(units[units.session.str.startswith("1001")][["session", "unit", "file_S1"]].to_string(index=False))
for n in sorted(names):
    f = n.split("/")[-1]
    if f.startswith("1001") and n.startswith("data/DAT/"):
        txt = z.read(n)[:20000].decode("latin1")
        print(f"  {f}: date {hc20.dat_value(txt, 'Date')}, time {' '.join(hc20.dat_value(txt, 'Time'))}")


**Result: partially reproduced.** The dataset holds 115 head-direction cells from 13 rats, the
paper's count; the per-rat counts match Table S1 except R029 (40 against 41) and R523 (2
against 1). Under the paper's criteria 110 of 115 pass in stable1. The 353 MEC cells and 556
MEC pairs of Table S1 cannot be counted, as only the head-direction cells are deposited. The
44 sessions give 146 simultaneously recorded head-direction pairs, against the paper's 142. The
two R523 units come from one session recorded on 1 October 2014 and the three units of 1001R029
from a session of 1 October 2013 (tracker headers), so the files do not show a unit filed under
the wrong rat; the TSD headers carry no date or rat to check further.

## hd-cell-rate

> The head-direction cells have long-duration spikes and a mean firing rate of 1.41 ± 0.20 AP/s.

Mean over the cells of spike count over trial duration, (a) stable1, (b) the mean of the three
trials. Spike width is not tested (no waveforms in the dataset).

In [ ]:
ra = UM.rate_S1
rb = UM[["rate_S1", "rate_RR", "rate_S2"]].mean(axis=1)
print(f"(a) stable1: mean {ra.mean():.3f} AP/s, SEM {ra.sem():.3f}, median {ra.median():.3f}, range {ra.min():.3f} to {ra.max():.3f}, n = {len(ra)}")
print(f"(b) mean of three trials: mean {rb.mean():.3f} AP/s, SEM {rb.sem():.3f}")
print("expected 1.41 +- 0.20; within tolerance:", abs(ra.mean() - 1.41) <= 0.2, abs(rb.mean() - 1.41) <= 0.2)
print("by rat (stable1 mean):", UM.groupby("rat").rate_S1.mean().round(2).to_dict())
print(f"clean variant, mean of three trials over units with all trials clean: {rb[clean('RR') & clean('S2')].mean():.3f} (n = {int((clean('RR') & clean('S2')).sum())})")

**Result: reproduced.** The mean stable1 rate is 1.392 AP/s (SEM 0.207) and the mean over the
three trials 1.433 AP/s, both within 1.41 ± 0.20.

## fig1c-firing-rate

> Firing rates of the head-direction cells were maintained across stable1, rotating and stable2.

A claim of absence on the paper's test: repeated-measures ANOVA on the three trials, with the
Greenhouse-Geisser correction when Mauchly's test is significant (STAR Methods, Statistics).

In [ ]:
Y = UM[["rate_S1", "rate_RR", "rate_S2"]].to_numpy()
a = hc20.rm_anova(Y)
print("as deposited:", hc20.fmt_anova(a))
print("means S1, rotating, S2 (AP/s):", np.round(a["means"], 3), "; mean change rotating - S1:", round(a["means"][1] - a["means"][0], 3),
      "; S2 - S1:", round(a["means"][2] - a["means"][0], 3))
print("paper: F(1.34, 153.21) = 1.89, p = 0.17")
Yc = Y.copy(); Yc[~clean("RR"), 1] = np.nan; Yc[~clean("S2"), 2] = np.nan
print("clean variant:", hc20.fmt_anova(hc20.rm_anova(Yc)))
print("log rates (a variant for the skewed rates):", hc20.fmt_anova(hc20.rm_anova(np.log(Y + 0.01))))
for r_, g in UM.groupby("rat"):
    if len(g) >= 5:
        print(f"  rat {r_} (n = {len(g)}): mean rates", g[["rate_S1", "rate_RR", "rate_S2"]].mean().round(2).tolist())

**Result: reproduced.** The repeated-measures ANOVA on rate over the three trials gives
F(1.35, 154.10) = 1.76, p = 0.184 (paper F(1.34, 153.21) = 1.89, p = 0.17); the mean rate is
1.392, 1.71 and 1.198 AP/s; the rotating mean includes one unit at 40.1 AP/s (1001R523 P9C1).
The clean variant gives p = 0.186. On log rates the trials differ
(p = 0.00149), a variant the paper does not use.

## fig1c-directionality

> Directional tuning of head-direction cells was strongly degraded during rotation in both
> frames, more in the arena frame than in the room frame, and returned in stable2
> (stable1 = stable2 > rotating room frame > rotating arena frame).

Rayleigh vector length of the 10 degree tuning curve per cell in four conditions;
repeated-measures ANOVA and Tukey-Kramer HSD (criterion).

Assumption: the Tukey test uses the error term of the repeated-measures ANOVA (`hc20.tukey_rm`).
The paper names Tukey-Kramer HSD after its repeated-measures ANOVAs (STAR Methods, Statistics;
Figure 1 legend, "ANOVAs on data in (C) and (D): Greenhouse-Geisser correction with
repeated-measures, Tukey-Kramer HSD tests") without saying which error term. Each cell is
measured in every condition, and the within-cell error is the one the ANOVA it follows uses, so
it is taken as the main test. The between-groups Tukey-Kramer test, which ignores the pairing, is
shown beside it; the outcome of this claim depends on the choice.

In [ ]:
conds = ["S1", "S2", "RR", "RA"]
def dir_tests(df, label):
    Y = df[[f"dir_{c}" for c in conds]].to_numpy()
    a = hc20.rm_anova(Y)
    # the paper prints uncorrected degrees of freedom, F(3, 342)
    print(f"{label}: {hc20.fmt_anova(a)}; uncorrected p = {stats.f.sf(a['F'], 3, 3 * (a['n'] - 1)):.3g}")
    print("  means S1, S2, RR, RA:", np.round(a["means"], 3), " medians:", np.round(np.nanmedian(Y, 0), 3))
    ok_ = ~np.isnan(Y).any(axis=1)
    trm = hc20.tukey_rm(Y, conds)
    tk = hc20.tukey({c: Y[ok_, i] for i, c in enumerate(conds)})
    trm["p_between_groups_tukey"] = [tk_p(tk, a_, b_)[0] for a_, b_ in zip(trm.group1, trm.group2)]
    print("  Tukey HSD, repeated-measures error (main) and between-groups (variant); meandiff = group2 - group1")
    print(trm.round(4).to_string(index=False))
    return trm
def tk_p(tk, a_, b_):
    r = tk[((tk.group1 == a_) & (tk.group2 == b_)) | ((tk.group1 == b_) & (tk.group2 == a_))].iloc[0]
    return float(r["p-adj"]), float(r.meandiff) * (1 if r.group1 == a_ else -1)
trm_dir = dir_tests(UM, "as deposited")
dc = UM.copy()
for c in conds:
    dc.loc[~clean(c), f"dir_{c}"] = np.nan
dir_tests(dc, "clean variant")
# paired differences per rat (room - arena)
print("per rat, median RR - RA:", UM.assign(d=UM.dir_RR - UM.dir_RA).groupby("rat").d.median().round(3).to_dict())

**Result: reproduced, resting on an assumption.** Directionality differs across conditions
(F(2.77, 315.60) = 72.66, p = 8.88e-34; means 0.46, 0.421, 0.344, 0.17 for stable1, stable2,
room, arena). With the Tukey test on the repeated-measures error term (the assumption above),
stable1 and stable2 do not differ (p = 0.2652), each stable trial is above the room frame
(p < 0.0001 and p = 0.0019) and the room frame above the arena frame (p < 0.0001), the order of
the legend at p < 0.01. With the between-groups Tukey test, stable2 against the room frame has
p = 0.0553, and the claim would be partially reproduced. In the clean variant the room frame is
not below stable2 (p = 0.4225) and is below stable1 only at p = 0.0407.

## fig1d-preference-change and fig1d-instability

> The change in preferred direction differs across trial pairs (stable1-stable2 =
> stable1-rotating room frame > stable1-rotating arena frame), and is greater for every trial
> pair than between the two halves of stable1.

> Directional instability (the KLD between tuning curves) differs across trial pairs and is
> greater for every trial pair than between the two halves of stable1.

Preference change: absolute circular difference of the bins of maximum rate. KLD: resistor
average of the two Kullback-Leibler divergences between Krichevsky-Trofimov probabilities,
from (a) spike counts per bin, (b) rates per bin. Tests: repeated-measures ANOVA on the three
trial pairs with Tukey-Kramer HSD, and a one-way ANOVA of the four groups with Dunnett's test
against the halves (criterion). The Tukey test is the repeated-measures one, as for
directionality; the between-groups test is shown beside it and gives the same order here.

In [ ]:
PAIRS_T = [("S1", "S2"), ("S1", "RR"), ("S1", "RA")]
def pair_measures(df):
    out = pd.DataFrame({"key": df.key})
    for a_, b_ in PAIRS_T + [("H1", "H2")]:
        out[f"pc_{a_}{b_}"] = hc20.circ_absdiff_deg(df[f"pref_{a_}"], df[f"pref_{b_}"])
        out[f"kc_{a_}{b_}"] = [hc20.kld(x, y) for x, y in zip(df[f"cnt_{a_}"], df[f"cnt_{b_}"])]
        out[f"kr_{a_}{b_}"] = [hc20.kld(x, y) for x, y in zip(df[f"curve_{a_}"], df[f"curve_{b_}"])]
    return out
PM = pair_measures(UM)
PMc = PM.copy()
for a_, b_ in PAIRS_T:
    bad = ~(clean(a_) & clean(b_))
    for m in ("pc", "kc", "kr"):
        PMc.loc[bad, f"{m}_{a_}{b_}"] = np.nan

def fig1d_tests(P, m, label):
    cols = [f"{m}_{a_}{b_}" for a_, b_ in PAIRS_T]
    Y = P[cols].to_numpy()
    a = hc20.rm_anova(Y)
    print(f"{label} [{m}] three trial pairs: {hc20.fmt_anova(a)}")
    print("   means S1-S2, S1-RR, S1-RA, halves:", np.round(np.r_[a["means"], np.nanmean(P[f'{m}_H1H2'])], 3))
    ok_ = ~np.isnan(Y).any(axis=1)
    lab = ["S1S2", "S1RR", "S1RA"]
    trm = hc20.tukey_rm(Y, lab)
    tk = hc20.tukey({c: Y[ok_, i] for i, c in enumerate(lab)})
    trm["p_between_groups_tukey"] = [tk_p(tk, a_, b_)[0] for a_, b_ in zip(trm.group1, trm.group2)]
    print(trm.round(4).to_string(index=False))
    groups = [P[c].dropna().to_numpy() for c in cols]
    halves = P[f"{m}_H1H2"].dropna().to_numpy()
    F, p = stats.f_oneway(*groups, halves)
    dn = stats.dunnett(*groups, control=halves)
    dn1 = stats.dunnett(*groups, control=halves, alternative="greater")
    print(f"   one-way ANOVA of four groups: F(3, {sum(map(len, groups)) + len(halves) - 4}) = {F:.2f}, p = {p:.3g}; "
          f"Dunnett against halves, two-sided p (S1S2, S1RR, S1RA):", np.round(dn.pvalue, 4),
          "; one-sided (greater):", np.round(dn1.pvalue, 4))
    return a, trm, p, dn.pvalue
R1D = {}
for m in ("pc", "kc", "kr"):
    R1D[m] = fig1d_tests(PM, m, "as deposited")
for m in ("pc", "kc", "kr"):
    fig1d_tests(PMc, m, "clean variant")

**Result: partially reproduced.** Both ANOVAs are significant (three trial pairs
F(2.00, 228.00) = 17.80, p = 6.56e-08; with the halves F(3, 456) = 19.62, p = 5.55e-12), and
every trial pair exceeds the stable1 halves by Dunnett's test (two-sided p = 0.019, 0.0073 and
< 0.0001; mean changes 58.1, 60.3 and 91.3 degrees against 39.5). The order among trial pairs
is the reverse of the legend's for the arena frame: stable1-arena has the largest change
(Tukey p < 0.0001 against each of the others), where the legend has it smallest.

**Result: partially reproduced.** Both ANOVAs are significant (spike-count KLD: three trial
pairs p = 7.22e-05, with the halves p = 1.36e-08; rate KLD: p = 0.000842 and p = 0.00256), but
not every trial pair exceeds the halves by Dunnett's test: with counts, stable1-room has
p = 0.1274 (stable1-stable2 p = 0.0044, stable1-arena p < 0.0001); with rates, only
stable1-arena does (p = 0.0008).

## fig1c-speed

> Rotation did not change locomotor speed relative to stable1, measured in either frame, but
> speed was lower in stable2 than in the other conditions.

Mean speed per recording session in stable1, rotating room frame, rotating arena frame and
stable2, from the 30 Hz positions of the TSD files (one track per session and trial), in cm
with the tracker's 2.1333 pixels per cm. Readings: (a) displacement between successive samples,
(b) displacement over 1 s steps. Untracked samples (x = y = 0) are dropped. A filter of steps
faster than 100 cm/s (tracking jumps) is an assumption of this attempt that the paper does not
state; the result is shown with and without it. Repeated-measures ANOVA with Greenhouse-Geisser
correction and Tukey HSD (criterion), on the repeated-measures error term as for directionality
(an assumption), with the between-groups Tukey test beside it.

In [ ]:
def mean_speed(t, step, vmax=None):
    x = t["x"].astype(float); y = t["y"].astype(float)
    good = ~((t["x"] == 0) & (t["y"] == 0))
    dx, dy = x[step:] - x[:-step], y[step:] - y[:-step]
    ok_ = good[step:] & good[:-step]
    v = np.hypot(dx, dy)[ok_] / hc20.PIX_PER_CM / (step / t["rate"])
    if vmax is not None:
        v = v[v <= vmax * (1 if step == 1 else 1)]
    return v.mean()
sessions = sorted(units.session.unique())
SP = {}
for lab, step, vmax in [("a", 1, None), ("a_filtered", 1, 100), ("b", 30, None), ("b_filtered", 30, 100)]:
    SP[lab] = pd.DataFrame({tr: [mean_speed(trk[(s, tr)], step, vmax) for s in sessions] for tr in conds}, index=sessions)
for lab, df in SP.items():
    a = hc20.rm_anova(df[conds].to_numpy())
    print(f"reading {lab}: means (cm/s) S1, S2, RR, RA =", np.round(a["means"], 2), "\n   ", hc20.fmt_anova(a))
    print(hc20.tukey_rm(df[conds].to_numpy(), conds).round(4).to_string(index=False))
    if True:
        tkb = hc20.tukey({c: df[c].to_numpy() for c in conds})
        print("   between-groups Tukey p:", {f"{r.group1}-{r.group2}": round(float(r['p-adj']), 4) for _, r in tkb.iterrows()})
dfc = SP["a"].copy()
dfc.loc[["1211R039"] + R029_WARP, ["RR"]] = np.nan
dfc.loc[R029_WARP, ["RA"]] = np.nan
print("clean variant, reading a:", hc20.fmt_anova(hc20.rm_anova(dfc[conds].to_numpy())))
print(hc20.tukey_rm(dfc[conds].to_numpy(), conds).round(4).to_string(index=False))
print("speed range over sessions (reading a, cm/s):", SP["a"].min().round(1).to_dict(), SP["a"].max().round(1).to_dict())
print(f"sessions: {len(sessions)} (the paper's degrees of freedom imply about 38)")

**Result: reproduced, resting on an assumption.** Over 44 sessions speed differs across
conditions in every reading (p from 2.9e-15 to 3.27e-17) and stable2 is lowest (10.82 against
14.64 to 15.67 cm/s in reading a), below each other condition at p < 0.0001. In reading (a)
with the 100 cm/s jump filter, stable1, room and arena do not differ (Tukey p = 0.9097,
0.2654, 0.0658; between-groups 0.9628, 0.5485, 0.2733). Without the filter the room and arena
frames differ at p = 0.0481 with the repeated-measures Tukey test and not with the between-groups
test (p = 0.2739), and in reading (b), with or without the filter, at p = 0.0058 and 0.0768. So
the outcome holds in reading (a) with the filter under either Tukey test, and in both readings
under the between-groups test; with the repeated-measures test it fails in reading (b) and in
reading (a) without the filter. The clean variant gives the reading (a)
pattern, room against arena p = 0.1020.

## no-angular-velocity-tuning

> The MEC head-direction cells showed no systematic tuning of firing rate to angular head
> velocity.

Per cell in stable1, the Pearson correlation of the spike count per 30 Hz sample (as a rate)
with (a) signed and (b) unsigned angular velocity, the head direction change between
successive samples (criterion). The fraction of cells with p < 0.05 is compared with the 5%
expected by chance with a one-sided binomial test at p < 0.05. Successive samples are not
independent, so the per-cell p values are optimistic; a variant in 1 s bins is shown.

In [ ]:
def av_corr(key, bin_samples=1):
    t = hc20.track(D, key, "S1")
    n = len(t["d"]) // bin_samples * bin_samples
    cnt = np.bincount(spk[key]["S1"]["i"], minlength=len(t["d"]))[:n].astype(float)
    d = t["d"][:n]
    if bin_samples > 1:
        # circular mean direction per bin, then change between bins
        d = hc20.window_dirs(d, bin_samples)
        cnt = cnt.reshape(-1, bin_samples).sum(1)
    av = ((np.diff(d) + 180) % 360 - 180) * t["rate"] / bin_samples
    r_ = cnt[1:] * t["rate"] / bin_samples
    ok_ = ~np.isnan(av)
    out = {}
    for lab, x in (("signed", av[ok_]), ("unsigned", np.abs(av[ok_]))):
        rr, pp = stats.pearsonr(x, r_[ok_])
        out[f"r_{lab}"], out[f"p_{lab}"] = rr, pp
    return out
AV = pd.DataFrame([av_corr(k) for k in UM.key])
AV1 = pd.DataFrame([av_corr(k, 30) for k in UM.key])
for nm, df in (("30 Hz samples (criterion)", AV), ("1 s bins (variant)", AV1)):
    for lab in ("signed", "unsigned"):
        sig = int((df[f"p_{lab}"] < 0.05).sum())
        bt = stats.binomtest(sig, len(df), 0.05, alternative="greater")
        print(f"{nm}, {lab}: {sig} of {len(df)} cells p < 0.05 ({sig / len(df):.1%}); binomial p = {bt.pvalue:.3g}; "
              f"median r = {df[f'r_{lab}'].median():.4f}, r range {df[f'r_{lab}'].min():.3f} to {df[f'r_{lab}'].max():.3f}; "
              f"positive among significant: {int(((df[f'p_{lab}'] < 0.05) & (df[f'r_{lab}'] > 0)).sum())}")

**Result: not reproduced, resting on an assumption.** In 30 Hz samples 12 of 115 cells (10.4%)
correlate with signed angular velocity at p < 0.05 (binomial p = 0.0127) and 49 of 115 with
unsigned angular velocity (binomial p = 5.71e-33), more than chance under both readings. The
correlations are small (r from -0.028 to 0.020 signed, -0.068 to 0.040 unsigned) and split in
sign (6 of 12 and 26 of 49 positive). The test is the claims file's; the paper shows no data.

## fig3-hd-pair-correlations-preserved

> The sub-second temporal coupling of head-direction cell pairs (Kendall correlations of spike
> counts at 40, 250 and 1000 ms) was maintained across stable1, rotating and stable2.

Spike counts in 40, 250 and 1000 ms bins over each trial from the spike timestamps, Kendall's
tau per simultaneously recorded pair, Fisher z-transformed; one-way ANOVA of the three
conditions at each timescale (criterion). The Pearson r between the pairs' tau in stable1 and
in each other trial is shown as well.

In [ ]:
PAIRS = []
for s_, g in units.groupby("session"):
    ks = [tuple(x) for x in g[["session", "unit"]].values]
    PAIRS += list(itertools.combinations(ks, 2))
print("simultaneously recorded pairs:", len(PAIRS))
SCALES = (0.04, 0.25, 1.0)
def counts(key, tr, w):
    t = hc20.track(D, key, tr)
    dur = len(t["d"]) / t["rate"] * 1.001
    return np.histogram(spk[key][tr]["t"], bins=np.arange(0, dur + w, w))[0]
t_ = time.time()
rows = []
for k1, k2 in PAIRS:
    row = {"k1": k1, "k2": k2}
    for tr in ("S1", "RR", "S2"):
        for w in SCALES:
            c1, c2 = counts(k1, tr, w), counts(k2, tr, w)
            n = min(len(c1), len(c2))
            row[f"tau_{tr}_{w}"] = stats.kendalltau(c1[:n], c2[:n])[0]
    row["clean"] = all(((k, tr) not in EXCL) for k in (k1, k2) for tr in ("RR", "S2"))
    rows.append(row)
KT = pd.DataFrame(rows)
print(f"Kendall correlations computed in {time.time() - t_:.0f} s")
for lab, df in (("as deposited", KT), ("clean variant", KT[KT.clean])):
    for w in SCALES:
        zs = [np.arctanh(df[f"tau_{tr}_{w}"].dropna().clip(-0.999, 0.999)) for tr in ("S1", "RR", "S2")]
        F, p = stats.f_oneway(*zs)
        print(f"{lab}, {int(w * 1000)} ms: mean z S1, rot, S2 = {[round(z.mean(), 4) for z in zs]}; "
              f"F(2, {sum(map(len, zs)) - 3}) = {F:.3f}, p = {p:.3g}; r(tau S1, tau rot) = {stats.pearsonr(df[f'tau_S1_{w}'], df[f'tau_RR_{w}'])[0]:.3f}, "
              f"r(tau S1, tau S2) = {stats.pearsonr(df[f'tau_S1_{w}'], df[f'tau_S2_{w}'])[0]:.3f}")
print("pairs with an undefined tau (a silent cell):", int(KT.filter(like="tau_").isna().any(axis=1).sum()))

**Result: reproduced.** The Fisher-transformed Kendall correlations of 146 pairs do not differ
across the three trials at 40 ms (p = 0.906), 250 ms (p = 0.72) or 1000 ms (p = 0.617), and the
pairs' tau in stable1 correlate with those in the rotating trial (r = 0.959, 0.869, 0.832) and
stable2 (r = 0.965, 0.917, 0.868). The clean variant gives p = 0.925, 0.738, 0.664.

## fig2-decoding-stable-vs-rotating

> Head direction decoded from stable1 ensemble tuning is relatively accurate in stable2 and
> relatively inaccurate and abruptly changing in both frames during rotation, although for
> some ensembles room-frame decoding is similar to stable2; seven ensembles of at least five
> head-direction cells were analysed.

The decoder of HD_Max_likelihood.py (criterion): windows of 20 tracker samples (660 ms; 4
samples, 133 ms, as a variant), the window's direction the circular mean of its valid samples,
36 bins of 10 degrees, a Poisson mean count per cell and bin fitted on stable1, the summed
Poisson log likelihood maximized over bins, windows without spikes left undecoded. The
paper's code bins windows without any valid direction at 0 degrees during training; they are
dropped here. Error is the circular difference of decoded and actual direction; summaries are
(a) the median and (b) the mean absolute error per ensemble and condition. As a reference, the
same decoding against directions shifted by 5 min. The clean variant leaves out the unit-trials
shown wrong in the data check, which removes 1023R029 from the rotating trial.

In [ ]:
ENS = [s for s, n in units.groupby("session").size().items() if n >= 5]
print("ensembles of at least 5 simultaneously recorded head-direction cells:", len(ENS), ENS)
def ens_keys(s):
    return [tuple(x) for x in units[units.session == s][["session", "unit"]].values]
# ensembles whose rotating-trial files are all clean (the clean variant of every rotating-trial analysis)
ENS_CR = [s for s in ENS if not any((k, tr) in EXCL for k in ens_keys(s) for tr in ("RR", "RA"))]
print("ensembles with clean rotating-trial files:", len(ENS_CR), "; left out:", sorted(set(ENS) - set(ENS_CR)))
def decode_session(s, nsamp, test_tr, keys=None, shift=0):
    keys = keys or ens_keys(s)
    # cells whose test file holds the session's track (1121R157 P11C2 stable2 does not)
    keys = [k for k in keys if (s, test_tr, k[1]) not in trk]
    tr1 = trk[(s, "S1")]
    w1 = hc20.window_dirs(tr1["d"], nsamp)
    c1 = np.array([hc20.window_counts(spk[k]["S1"]["i"], len(tr1["d"]), nsamp) for k in keys])
    lam = hc20.fit_poisson_tuning(c1, w1)
    t2 = trk[(s, test_tr)]
    w2 = hc20.window_dirs(np.roll(t2["d"], shift), nsamp)
    c2 = np.array([hc20.window_counts(spk[k][test_tr]["i"], len(t2["d"]), nsamp) for k in keys])
    est = hc20.decode(lam, c2)
    err = hc20.signed_err(est, w2)
    return dict(est=est, true=w2, err=err, n_cells=len(keys), n_win=len(w2), n_dec=int(np.sum(~np.isnan(err))))
DEC = {}
rows = []
for nsamp in (20, 4):
    for s in ENS:
        for tr in ("S2", "RR", "RA"):
            r = decode_session(s, nsamp, tr)
            DEC[(nsamp, s, tr)] = r
            ref = decode_session(s, nsamp, tr, shift=int(300 * 30))
            e = np.abs(r["err"]); e = e[~np.isnan(e)]
            er = np.abs(ref["err"]); er = er[~np.isnan(er)]
            rows.append(dict(window=nsamp, session=s, trial=tr, cells=r["n_cells"], windows=r["n_win"], decoded=r["n_dec"],
                             median_err=np.median(e), mean_err=e.mean(), median_err_shift5min=np.median(er)))
DE = pd.DataFrame(rows)
for nsamp in (20, 4):
    d = DE[DE.window == nsamp]
    print(f"\nwindow {nsamp} samples:")
    print(d.pivot(index="session", columns="trial", values="median_err").round(1).rename(columns=lambda c: f"median_{c}").join(
          d.pivot(index="session", columns="trial", values="mean_err").round(1).rename(columns=lambda c: f"mean_{c}")).join(
          d.pivot(index="session", columns="trial", values="median_err_shift5min").round(1).rename(columns=lambda c: f"shift_{c}")).to_string())
    for m in ("median_err", "mean_err"):
        p = d.pivot(index="session", columns="trial", values=m)
        print(f"  [{m}] stable2 below arena frame in {int((p.S2 < p.RA).sum())} of {len(p)}; below room frame in {int((p.S2 < p.RR).sum())} of {len(p)}")
print("\ndecoded windows (window 20):", DE[DE.window == 20][["session", "trial", "decoded", "windows"]].pivot(index="session", columns="trial", values="decoded").to_dict())
d20 = DE[DE.window == 20].pivot(index="session", columns="trial", values="median_err")
ref20 = DE[DE.window == 20].pivot(index="session", columns="trial", values="median_err_shift5min")
print("\nstable2 median error minus the 5 min shifted reference (660 ms; near 0 or above means decoding at chance):",
      (d20.S2 - ref20.S2).round(1).to_dict())
print("ensembles with stable2 median error within 10 degrees of, or above, the shifted reference:",
      int(((d20.S2 - ref20.S2) > -10).sum()), "of", len(d20))

# clean variant: the unit-trials shown wrong in the data check are left out of the trial they affect
# (both training and test use the cells kept); an ensemble with no cell left in a trial is dropped
rows_c = []
for nsamp in (20, 4):
    for s in ENS:
        for tr in ("S2", "RR", "RA"):
            kc = [k for k in ens_keys(s) if (k, tr) not in EXCL]
            if not kc:
                continue
            r = decode_session(s, nsamp, tr, keys=kc)
            e = np.abs(r["err"]); e = e[~np.isnan(e)]
            rows_c.append(dict(window=nsamp, session=s, trial=tr, cells=r["n_cells"], median_err=np.median(e), mean_err=e.mean()))
DEc = pd.DataFrame(rows_c)
for nsamp in (20, 4):
    d = DEc[DEc.window == nsamp]
    for m in ("median_err", "mean_err"):
        p = d.pivot(index="session", columns="trial", values=m).dropna()
        print(f"clean variant, window {nsamp} [{m}]: {len(p)} ensembles; stable2 below arena frame in {int((p.S2 < p.RA).sum())} of {len(p)}; "
              f"below room frame in {int((p.S2 < p.RR).sum())} of {len(p)}")


**Result: reproduced, resting on an assumption.** There are seven ensembles of at least five
cells. With 660 ms windows, the stable2 error is below the arena-frame error in 4 of 7
ensembles and below the room-frame error in 5 of 7, for the median and the mean error alike;
the 133 ms windows give the same counts. The "majority of seven" threshold is the claims file's
assumption. The margin is thin. Stable2 decoding is at chance in 1026R157 and 1117R157 (median
error 1.4 and 0.0 degrees from the shifted reference) and near it in 1121R157 (11.7 degrees
below), where the paper reports one ensemble with poor stable2 decoding. In the clean variant,
without 1023R029, whose rotating tracks lag the spikes, the stable2 error is below the arena-frame
error in 3 of 6 ensembles and below the room-frame error in 4 of 6, under both measures and
windows, so the arena-frame count meets the threshold only with that session.

## fig5-error-inhomogeneous

> The angular distribution of decoding error is inhomogeneous in the room frame (resultant
> r = 0.589) and the arena frame (r = 0.653) and differs from the distribution of sampled head
> directions, which is itself biased in both frames (r = 0.147 room, r = 0.378 arena).

Decoding errors of the rotating trial in each frame (660 ms windows, 133 ms as a variant): a
Rayleigh test of uniformity, and Kuiper's test against the sampled head directions of the same
trial; the resultant r and Rayleigh test of the sampled directions. Readings: (a) the seven
ensembles pooled, (b) the exemplar ensemble with the most cells (four have six; the one with
the most decoded rotating windows is taken), (c) each ensemble apart, with the median r
(criterion). Variants: the clean variant (without 1023R029), and, for the question put to a
reader (questions/park-2019.yaml), a reading of the "polar distribution of directional error" as
a distribution over head direction: the head directions of the windows decoded within 15, 30 or
45 degrees, with the Rayleigh test and Kuiper's test against the sampled head directions.

In [ ]:
def fig5_stats(sess, nsamp):
    out = {}
    for tr in ("RR", "RA"):
        err = np.concatenate([DEC[(nsamp, s, tr)]["err"] for s in sess])
        err = np.radians(err[~np.isnan(err)])
        hd = np.concatenate([trk[(s, tr)]["d"] for s in sess])
        hd = np.radians(hd[~np.isnan(hd)])
        re_ = np.abs(np.mean(np.exp(1j * err)))
        rh = np.abs(np.mean(np.exp(1j * hd)))
        V, lam, pk = hc20.kuiper_two(err, hd)
        out[tr] = dict(n_err=len(err), r_err=re_, z_err=len(err) * re_**2, p_err=hc20.rayleigh_p(re_, len(err)),
                       mean_err_deg=np.degrees(np.angle(np.mean(np.exp(1j * err)))),
                       n_hd=len(hd), r_hd=rh, z_hd=len(hd) * rh**2, p_hd=hc20.rayleigh_p(rh, len(hd)),
                       kuiper_V=V, kuiper_p=pk)
    return out
dec_rot = DE[(DE.window == 20) & (DE.trial != "S2")].groupby("session").decoded.sum()
six = [s for s in ENS if len(ens_keys(s)) == 6]
EXEMPLAR = dec_rot[six].idxmax()
print("six-cell ensembles:", six, "; exemplar (most decoded rotating windows):", EXEMPLAR)
F5 = {}
for nsamp in (20, 4):
    F5[(nsamp, "a")] = fig5_stats(ENS, nsamp)
    F5[(nsamp, "b")] = fig5_stats([EXEMPLAR], nsamp)
    per = {s: fig5_stats([s], nsamp) for s in ENS}
    F5[(nsamp, "c")] = {tr: {k: np.median([per[s][tr][k] for s in ENS]) for k in per[ENS[0]][tr]} for tr in ("RR", "RA")}
    F5[(nsamp, "per")] = per
for (nsamp, rd), v in F5.items():
    if rd == "per":
        continue
    for tr in ("RR", "RA"):
        x = v[tr]
        print(f"window {nsamp}, reading {rd}, {tr}: error r = {x['r_err']:.3f} (n = {x['n_err']:.0f}, z = {x['z_err']:.1f}, p = {x['p_err']:.3g}, "
              f"mean error {x['mean_err_deg']:.0f} deg); HD r = {x['r_hd']:.3f} (n = {x['n_hd']:.0f}, z = {x['z_hd']:.1f}, p = {x['p_hd']:.3g}); "
              f"Kuiper V = {x['kuiper_V']:.3f}, p = {x['kuiper_p']:.3g}")
print("expected r: error room 0.589 +- 0.059, arena 0.653 +- 0.065; HD room 0.147 +- 0.015, arena 0.378 +- 0.038")
print("\nper ensemble (window 20): r_err RR, r_err RA, r_hd RR, r_hd RA")
for s in ENS:
    x = F5[(20, "per")][s]
    print(f"  {s}: {x['RR']['r_err']:.3f} {x['RA']['r_err']:.3f} {x['RR']['r_hd']:.3f} {x['RA']['r_hd']:.3f}; Kuiper p {x['RR']['kuiper_p']:.2g} {x['RA']['kuiper_p']:.2g}")

# clean variant: the ensembles with clean rotating-trial files pooled (a), and their median (c)
for nsamp in (20, 4):
    xa = fig5_stats(ENS_CR, nsamp)
    xc = {tr: {k: np.median([F5[(nsamp, "per")][s][tr][k] for s in ENS_CR]) for k in ("r_err", "p_err", "r_hd", "kuiper_p")} for tr in ("RR", "RA")}
    for tr in ("RR", "RA"):
        print(f"clean variant ({len(ENS_CR)} ensembles), window {nsamp}, {tr}: (a) error r = {xa[tr]['r_err']:.3f} (p = {xa[tr]['p_err']:.3g}), "
              f"HD r = {xa[tr]['r_hd']:.3f}, Kuiper p = {xa[tr]['kuiper_p']:.3g}; (c) median error r = {xc[tr]['r_err']:.3f}, "
              f"median HD r = {xc[tr]['r_hd']:.3f}")

# variant for the question put to a reader (questions/park-2019.yaml): the "polar distribution of
# directional error" read as a distribution over head direction, the head directions of the
# accurately decoded windows (|error| at most 15, 30 or 45 degrees), with the Rayleigh test and
# Kuiper's test against the sampled head directions of the trial
def hd_accurate(sess, nsamp, tr, thr):
    err = np.concatenate([DEC[(nsamp, s, tr)]["err"] for s in sess])
    true = np.concatenate([DEC[(nsamp, s, tr)]["true"] for s in sess])
    acc = ~np.isnan(err) & ~np.isnan(true) & (np.abs(err) <= thr)
    a = np.radians(true[acc])
    hd = np.concatenate([trk[(s, tr)]["d"] for s in sess])
    hd = np.radians(hd[~np.isnan(hd)])
    r = np.abs(np.mean(np.exp(1j * a)))
    return dict(n=len(a), r=r, p=hc20.rayleigh_p(r, len(a)), kuiper_p=hc20.kuiper_two(a, hd)[2],
                mean_deg=np.degrees(np.angle(np.mean(np.exp(1j * a)))) % 360)
print("\nhead-direction reading (variant): head directions of windows decoded within the threshold")
for nsamp in (20, 4):
    for thr in (15, 30, 45):
        for tr in ("RR", "RA"):
            xa, xb = hd_accurate(ENS, nsamp, tr, thr), hd_accurate([EXEMPLAR], nsamp, tr, thr)
            per = [hd_accurate([s], nsamp, tr, thr) for s in ENS]
            print(f"  window {nsamp}, |error| <= {thr}, {tr}: (a) n = {xa['n']}, r = {xa['r']:.3f}, p = {xa['p']:.3g}, Kuiper p = {xa['kuiper_p']:.3g}; "
                  f"(b) n = {xb['n']}, r = {xb['r']:.3f}, p = {xb['p']:.3g}, Kuiper p = {xb['kuiper_p']:.3g}; "
                  f"(c) median r = {np.median([x['r'] for x in per]):.3f}, median Rayleigh p = {np.median([x['p'] for x in per]):.3g}")
print("expected r (claims file): room 0.589 +- 0.059, arena 0.653 +- 0.065")


**Result: partially reproduced.** The decoding errors are not uniform and differ from the
sampled directions in readings (b) and (c) in both frames (Rayleigh p from 2.66e-10 to 0.00459,
Kuiper p < 1e-26), and the sampled directions are non-uniform; in reading (a) the arena-frame
error is not (p = 0.0626). This meets the criterion's "tests hold but not every r" under (b) and
(c), but none of the four resultant lengths is near the paper's under any reading or window:
error r 0.172 and 0.014 (a), 0.050 and 0.144 (b), 0.139 and 0.100 (c), against 0.589 and 0.653;
head-direction r 0.035 and 0.276 (a) against 0.147 and 0.378. One ensemble, 0930R029, has a
room-frame error r of 0.560. The clean variant changes little (error r 0.177 and 0.014 pooled).
Under the head-direction reading of the question to a reader, the head directions of accurately
decoded windows are non-uniform and differ from the sampled ones in every case (Rayleigh p at
most 0.0036), with r from 0.104 to 0.550; the largest, 0.529 to 0.550, are in the arena frame at
133 ms (exemplar, and median over ensembles), below 0.653.

## fig5c-error-minima-at-cues

> The room-frame decoding error histogram has strong minima at 110 and 350 degrees in the room
> frame, the directions of the leading edges of prominent orienting cues given the clockwise
> rotation; the minima correspond to the cue edges in the room frame but not in the arena frame.

Readings (criterion): (a) the mean absolute decoding error as a function of the rat's room-frame
head direction in 10 degree bins, (b) the histogram of the signed room-frame decoding error
(0 to 360 degrees) in 10 degree bins; under the pooling readings of fig5. The two deepest local
minima (circular) are reported, at bin centres. For reading (a), pooled, 660 ms, the whole curve,
the minima after the first two, a null with each ensemble's errors circularly shifted against its
head directions, and the clean variant are shown.

In [ ]:
def two_minima(v):
    v = np.asarray(v, float)
    loc = [i for i in range(len(v)) if v[i] < v[i - 1] and v[i] < v[(i + 1) % len(v)]]
    loc = sorted(loc, key=lambda i: v[i])[:2]
    return sorted([(i * 10 + 5) for i in loc])
F5C = {}
for nsamp in (20, 4):
    for rd, sess in (("a", ENS), ("b", [EXEMPLAR])):
        err = np.concatenate([DEC[(nsamp, s, "RR")]["err"] for s in sess])
        true = np.concatenate([DEC[(nsamp, s, "RR")]["true"] for s in sess])
        ok_ = ~np.isnan(err)
        b = hc20.bin_index(true[ok_])
        mae = np.array([np.mean(np.abs(err[ok_][b == i])) if np.any(b == i) else np.nan for i in range(36)])
        hist = np.bincount(hc20.bin_index(err[ok_] % 360), minlength=36)
        F5C[(nsamp, rd)] = dict(mae=two_minima(mae), hist=two_minima(hist))
        print(f"window {nsamp}, pooling {rd}: (a) minima of mean |error| by room HD at {two_minima(mae)} deg "
              f"(values {np.round(np.sort(mae)[:2], 1)}, range {np.nanmin(mae):.1f} to {np.nanmax(mae):.1f}); "
              f"(b) minima of the signed error histogram at {two_minima(hist)} deg (counts range {hist.min()} to {hist.max()})")
    per = {s: two_minima(np.bincount(hc20.bin_index(DEC[(nsamp, s, 'RR')]['err'][~np.isnan(DEC[(nsamp, s, 'RR')]['err'])] % 360), minlength=36)) for s in ENS}
    print(f"   reading (c), per ensemble, histogram minima: {per}")
print("expected minima 110 and 350 deg, +- 10")

# robustness of reading (a), seven ensembles pooled, 660 ms: the whole curve, the third-deepest
# minimum, and a null in which each ensemble's errors are circularly shifted against its head
# directions (1000 shifts of at least a tenth of the windows; seeded generator for this section)
def mae_curve(sess, nsamp=20, shift_rng=None):
    errs, bins_ = [], []
    for s in sess:
        e = DEC[(nsamp, s, "RR")]["err"]; t = DEC[(nsamp, s, "RR")]["true"]
        ok_ = ~np.isnan(e) & ~np.isnan(t)
        e, t = e[ok_], t[ok_]
        if shift_rng is not None:
            e = np.roll(e, shift_rng.integers(len(e) // 10, len(e) - len(e) // 10))
        errs.append(np.abs(e)); bins_.append(hc20.bin_index(t))
    e, b = np.concatenate(errs), np.concatenate(bins_)
    return np.array([e[b == i].mean() if np.any(b == i) else np.nan for i in range(36)])
def all_minima(v):
    loc = [i for i in range(len(v)) if v[i] < v[i - 1] and v[i] < v[(i + 1) % len(v)]]
    return sorted([(i * 10 + 5, round(float(v[i]), 1)) for i in loc], key=lambda x: x[1])
def matches(m2):
    return len(m2) == 2 and abs(m2[0] - 110) <= 10 and abs(m2[1] - 350) <= 10
cur = mae_curve(ENS)
print("\nmean |error| (deg) by room head direction, 10 degree bins from 0 (pooled, 660 ms):", np.round(cur, 1).tolist())
print("local minima, deepest first (centre, value):", all_minima(cur))
def depth2(v):  # the two deepest local minima below the curve's median
    return np.nanmedian(v) - np.mean([m[1] for m in all_minima(v)[:2]])
depth = depth2(cur)
rng5c = np.random.default_rng(20195)
null_match, null_depth = [], []
for _ in range(1000):
    v = mae_curve(ENS, shift_rng=rng5c)
    null_match.append(matches(two_minima(v)))
    null_depth.append(depth2(v))
null_depth = np.array(null_depth)
print(f"null (errors shifted against head direction): two deepest minima within 10 degrees of 110 and 350 in {np.mean(null_match):.3f} of shifts; "
      f"depth of the two minima below the curve's median {depth:.1f} deg, null median {np.median(null_depth):.1f} (95th percentile "
      f"{np.percentile(null_depth, 95):.1f}), fraction of null at least as deep {np.mean(null_depth >= depth):.3f}")
cur_c = mae_curve(ENS_CR)
print(f"clean variant ({len(ENS_CR)} ensembles pooled, 660 ms): minima {two_minima(cur_c)}, deepest first {all_minima(cur_c)[:3]}")
for nsamp in (20, 4):
    errc = np.concatenate([DEC[(nsamp, s, "RR")]["err"] for s in ENS_CR])
    print(f"clean variant, window {nsamp}: (a) minima {two_minima(mae_curve(ENS_CR, nsamp))}; (b) histogram minima "
          f"{two_minima(np.bincount(hc20.bin_index(errc[~np.isnan(errc)] % 360), minlength=36))}")


**Result: reproduced.** Under reading (a) with the seven ensembles pooled and 660 ms windows,
the two deepest minima of the mean absolute error by room head direction are at 105 and 345
degrees (63.2 and 63.9), within 10 degrees of 110 and 350. The third-deepest is at 325 degrees
(65.2), separated from 345 by one bin of 65.4. The two minima lie 13.8 degrees below the curve's
median, against a median of 7.5 (95th percentile 10.2, none of 1000 as deep) when the errors are shifted against
head direction, and no shift of 1000 puts the minima at 110 and 350. The clean variant (six ensembles)
puts them at 115 and 345. The other readings and windows put the minima elsewhere (for example 75
and 115 degrees for the signed error histogram, 235 and 335 degrees for reading (a) at 133 ms),
so the result holds under one reading and window only.

## figs2-frame-preference

> During rotation, frame-specific directional discharge alternated between signaling room and
> arena direction; four of seven ensembles showed an overall room preference and two of seven
> an arena preference.

Per cell and 133 ms interval (4 tracker samples) of the rotating trial, Idir =
|p(n|d) log2(p(n|d)/p(n))| for the observed count n and the interval's direction bin d
(circular mean of its samples), with room and with arena directions; ΔIdir is the room sum
over cells minus the arena sum (STAR Methods, Estimating momentary directional information).
Intervals without a valid direction in both frames are dropped. Open details and readings
(criterion): direction bins of (10) 10 degrees or (45) 45 degrees; probabilities from (rot)
the rotating trial itself or (s1) stable1; the SD of the 2 SD rule, (sdD) the SD of the
ensemble's ΔIdir series or (sdS1) the SD of the ensemble's summed Idir in stable1; the margin
for "no preference", (f2) the two probabilities within a factor of 2 or (p05) both below 0.05.

In [ ]:
NS = 4  # 133 ms
def idir_ensemble(s, nbins, src):
    keys = ens_keys(s)
    out = {}
    tr1 = trk[(s, "S1")]
    w1 = hc20.bin_index(hc20.window_dirs(tr1["d"], NS), nbins)
    c1 = [hc20.window_counts(spk[k]["S1"]["i"], len(tr1["d"]), NS) for k in keys]
    for fr in ("RR", "RA"):
        t = trk[(s, fr)]
        w = hc20.bin_index(hc20.window_dirs(t["d"], NS), nbins)
        I = []
        for k, cs1 in zip(keys, c1):
            c = hc20.window_counts(spk[k][fr]["i"], len(t["d"]), NS)
            if src == "rot":
                I.append(hc20.idir_series(c, w, nbins=nbins))
            else:
                I.append(hc20.idir_series(c, w, cs1, w1, nbins=nbins))
        I = np.array(I)
        I[:, w < 0] = np.nan
        out[fr] = I
    # stable1 summed Idir, for the sdS1 reading
    I1 = np.array([hc20.idir_series(cs1, w1, nbins=nbins) for cs1 in c1])
    out["S1sum"] = np.nansum(I1, 0)[w1 >= 0]
    n = min(out["RR"].shape[1], out["RA"].shape[1])
    room, arena = out["RR"][:, :n], out["RA"][:, :n]
    valid = ~np.isnan(room).all(0) & ~np.isnan(arena).all(0)
    out["room_sum"] = np.where(valid, np.nansum(room, 0), np.nan)
    out["arena_sum"] = np.where(valid, np.nansum(arena, 0), np.nan)
    out["dI"] = out["room_sum"] - out["arena_sum"]
    return out
IDIR = {(s, nb, src): idir_ensemble(s, nb, src) for s in ENS for nb in (36, 8) for src in ("rot", "s1")}

def classify(pp, pm, margin):
    if margin == "f2":
        return "room" if pp > 2 * pm else ("arena" if pm > 2 * pp else "none")
    if pp < 0.05 and pm < 0.05:
        return "none"
    return "room" if pp > pm else ("arena" if pm > pp else "none")
rows = []
for (s, nb, src), o in IDIR.items():
    dI = o["dI"][~np.isnan(o["dI"])]
    for sdr in ("sdD", "sdS1"):
        sd = dI.std() if sdr == "sdD" else o["S1sum"].std()
        pp, pm = np.mean(dI > 2 * sd), np.mean(dI < -2 * sd)
        for mg in ("f2", "p05"):
            rows.append(dict(session=s, bins=360 // nb, src=src, sd_rule=sdr, margin=mg, n=len(dI), mean_dI=dI.mean(),
                             sd=sd, p_plus=pp, p_minus=pm, pref=classify(pp, pm, mg)))
FP = pd.DataFrame(rows)
summ = FP.groupby(["bins", "src", "sd_rule", "margin"]).pref.apply(lambda x: f"room {int((x == 'room').sum())}, arena {int((x == 'arena').sum())}, none {int((x == 'none').sum())}")
print("counts of ensembles by reading (expected room 4, arena 2):")
print(summ.to_string())
match = FP.groupby(["bins", "src", "sd_rule", "margin"]).pref.apply(lambda x: (x == "room").sum() == 4 and (x == "arena").sum() == 2)
print("\nreadings that give 4 room and 2 arena:", match[match].index.tolist())
more_room = FP.groupby(["bins", "src", "sd_rule", "margin"]).pref.apply(lambda x: (x == "room").sum() > (x == "arena").sum())
print("readings with more room- than arena-preferring ensembles:", int(more_room.sum()), "of", len(more_room))
print("\nper ensemble, 10 degree bins, rotating-trial probabilities:")
print(FP[(FP.bins == 10) & (FP.src == "rot") & (FP.margin == "f2")][["session", "sd_rule", "n", "mean_dI", "sd", "p_plus", "p_minus", "pref"]].round(4).to_string(index=False))
print("\nper ensemble, the matching reading (45 degree bins, rotating-trial probabilities, SD of ΔIdir, both below 0.05):")
print(FP[(FP.bins == 45) & (FP.src == "rot") & (FP.sd_rule == "sdD") & (FP.margin == "p05")][["session", "n", "mean_dI", "sd", "p_plus", "p_minus", "pref"]].round(4).to_string(index=False))

FPc = FP[FP.session.isin(ENS_CR)]
print(f"\nclean variant ({len(ENS_CR)} ensembles, without {sorted(set(ENS) - set(ENS_CR))}):")
print(FPc.groupby(["bins", "src", "sd_rule", "margin"]).pref.apply(lambda x: f"room {int((x == 'room').sum())}, arena {int((x == 'arena').sum())}, none {int((x == 'none').sum())}").to_string())
# sensitivity of the matching reading to the "both below" margin
m_ = FP[(FP.bins == 45) & (FP.src == "rot") & (FP.sd_rule == "sdD") & (FP.margin == "p05")]
for thr in (0.03, 0.04, 0.045, 0.05, 0.055, 0.06, 0.07):
    pr_ = ["none" if (pp < thr and pm < thr) else ("room" if pp > pm else "arena") for pp, pm in zip(m_.p_plus, m_.p_minus)]
    print(f"  45 degree bins, rotating-trial probabilities, SD of ΔIdir, both below {thr}: room {pr_.count('room')}, arena {pr_.count('arena')}")


**Result: reproduced, resting on an assumption.** One of the 16 readings, 45 degree bins with
probabilities from the rotating trial, the SD of the ΔIdir series and the "both below 0.05"
margin, gives 4 room-preferring and 2 arena-preferring ensembles. 14 of 16 readings give more
room- than arena-preferring ensembles, and the others give 0 to 5 room and 0 or 1 arena. The
match is fragile. Two ensembles are classed on probabilities near 0.05 (1117R157 room, 0.0539
against 0.0303; 1121R157 arena, 0.0413 against 0.0606): with the margin at 0.04 to 0.05 the counts
are 4 and 2, at 0.055 or 0.06 they are 3 and 2, at 0.07 3 and 1. The other arena ensemble is
1023R029, whose rotating tracks lag the spikes; in the clean variant (six ensembles) the matching
reading gives 4 room and 1 arena, and no reading gives 4 and 2.

## fig2c-proximity-anticorrelation

> The rat's relative proximity to one shock zone versus the other is anti-correlated with
> frame-specific directional information, so that near the room shock zone directional
> discharge favours the arena frame and the reverse; this holds for Idir in each frame (mean
> r = -0.141 ± 0.029, all p < 0.0001) and for ΔIdir (r = -0.18, p < 0.05, in the exemplar).

At the same 133 ms steps, the distance d_room of the room-frame position (RA_rm track) to the
room shock zone and d_arena of the arena-frame position (RA_ar track) to the arena shock zone,
each zone taken as the centre point of its sector from the tracker header (an assumption: the
paper does not say how distance to a zone is measured). Readings of relative proximity
(criterion): (a) (d_arena - d_room) / (d_arena + d_room), (b) d_arena - d_room, positive when
nearer the room zone. The room-frame Idir sum is paired with proximity to the room zone and
the arena-frame Idir sum with its mirror, giving 14 correlations; ΔIdir is paired with
proximity to the room zone. Pearson correlations, under each Idir reading of figs2.

In [ ]:
def zone_point(s, frame):
    r = DAT[(DAT.session == s) & (DAT.frame == frame)].iloc[0]
    rc = (r.r_in + r.r_out) / 2
    th = np.radians(r.center)
    return 127.5 + rc * np.cos(th), 127.5 - rc * np.sin(th)  # y grows downwards (checked above)
def mean_pos(t, nsamp):
    n = len(t["x"]) // nsamp * nsamp
    x = t["x"][:n].astype(float); y = t["y"][:n].astype(float)
    bad = (t["x"][:n] == 0) & (t["y"][:n] == 0)
    x[bad] = np.nan; y[bad] = np.nan
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # windows with no tracked sample give nan
        return np.nanmean(x.reshape(-1, nsamp), 1), np.nanmean(y.reshape(-1, nsamp), 1)
PROX = {}
for s in ENS:
    xr, yr = mean_pos(trk[(s, "RR")], NS)
    xa, ya = mean_pos(trk[(s, "RA")], NS)
    zr, za = zone_point(s, "Room"), zone_point(s, "Arena")
    n = min(len(xr), len(xa))
    d_room = np.hypot(xr[:n] - zr[0], yr[:n] - zr[1])
    d_arena = np.hypot(xa[:n] - za[0], ya[:n] - za[1])
    PROX[s] = {"a": (d_arena - d_room) / (d_arena + d_room), "b": d_arena - d_room}
rows = []
for (s, nb, src), o in IDIR.items():
    for pr in ("a", "b"):
        p = PROX[s][pr]
        n = min(len(p), len(o["dI"]))
        for lab, y, x in (("room", o["room_sum"][:n], p[:n]), ("arena", o["arena_sum"][:n], -p[:n]), ("dI", o["dI"][:n], p[:n])):
            m = ~np.isnan(x) & ~np.isnan(y)
            r_, pv = stats.pearsonr(x[m], y[m])
            rows.append(dict(session=s, bins=360 // nb, src=src, prox=pr, what=lab, r=r_, p=pv, n=int(m.sum())))
PC = pd.DataFrame(rows)
g = PC[PC.what != "dI"].groupby(["bins", "src", "prox"])
res = g.apply(lambda d: pd.Series(dict(mean_r=d.r.mean(), sem_r=d.r.sem(), n_neg=int((d.r < 0).sum()),
                                       n_neg_p1e4=int(((d.r < 0) & (d.p < 1e-4)).sum()), n=len(d))))
res_d = PC[PC.what == "dI"].groupby(["bins", "src", "prox"]).apply(lambda d: pd.Series(dict(mean_r_dI=d.r.mean(), n_neg_dI=int((d.r < 0).sum()), n_neg_p05_dI=int(((d.r < 0) & (d.p < 0.05)).sum()))))
print("frame-specific Idir against relative proximity, 14 correlations per reading (expected mean r -0.141 +- 0.029 or -0.132 +- 0.024):")
print(res.join(res_d).round(4).to_string())
print("\nper correlation, 10 degree bins, rotating-trial probabilities, proximity (a):")
print(PC[(PC.bins == 10) & (PC.src == "rot") & (PC.prox == "a")].pivot(index="session", columns="what", values="r").round(4).to_string())
print("exemplar", EXEMPLAR, "ΔIdir r:", PC[(PC.session == EXEMPLAR) & (PC.what == "dI")][["bins", "src", "prox", "r", "p"]].round(4).to_string(index=False))

resc = PC[(PC.what != "dI") & PC.session.isin(ENS_CR)].groupby(["bins", "src", "prox"]).apply(
    lambda d: pd.Series(dict(mean_r=d.r.mean(), n_neg=int((d.r < 0).sum()), n_neg_p1e4=int(((d.r < 0) & (d.p < 1e-4)).sum()), n=len(d))))
print(f"\nclean variant ({len(ENS_CR)} ensembles):")
print(resc.round(4).to_string())


**Result: partially reproduced.** Under every reading the mean of the 14 frame-specific
correlations is negative (-0.052 to -0.088) but outside both expected ranges (-0.141 ± 0.029,
-0.132 ± 0.024); 11 or 12 of 14 are negative and 7 to 10 negative at p < 0.0001. The ΔIdir
correlations average -0.055 to -0.128 (5 or 6 of 7 negative), within -0.132 ± 0.024 under the
stable1-probability readings with 45 degree bins (-0.126, -0.128). In the clean variant (12
correlations) the mean is -0.049 to -0.119, within both ranges under those two readings (-0.113,
-0.119), but only 6 to 10 of 12 are negative at p < 0.0001 (9 to 11 negative), so the outcome
is the same.

## figs3-no-head-direction-bias

> The frame preference ΔIdir does not vary systematically with head direction; its polar
> distribution over head direction in either frame is not biased toward the shock zones and
> does not differ from uniform.

Per ensemble and frame, the mean ΔIdir in 10 degree head-direction bins of that frame,
normalized to 0 to 1 by adding the magnitude of the minimum and dividing by the maximum
(Figure S3 legend), and a Rayleigh test with the normalized values as weights; n is (a) the
number of bins, (b) the sum of the weights (criterion). Under each Idir reading of figs2.

In [ ]:
rows = []
for (s, nb, src), o in IDIR.items():
    for fr in ("RR", "RA"):
        t = trk[(s, fr)]
        w = hc20.bin_index(hc20.window_dirs(t["d"], NS), 36)
        n = min(len(w), len(o["dI"]))
        w, dI = w[:n], o["dI"][:n]
        m = (w >= 0) & ~np.isnan(dI)
        mean_b = np.array([dI[m][w[m] == i].mean() if np.any(w[m] == i) else np.nan for i in range(36)])
        v = mean_b + abs(np.nanmin(mean_b))
        v = v / np.nanmax(v)
        ok_ = ~np.isnan(v)
        ang = hc20.centers(36)[ok_]
        r_ = np.abs(np.sum(v[ok_] * np.exp(1j * ang))) / np.sum(v[ok_])
        rows.append(dict(session=s, bins=360 // nb, src=src, frame=fr, r=r_, p_bins=hc20.rayleigh_p(r_, int(ok_.sum())),
                         p_wsum=hc20.rayleigh_p(r_, float(np.sum(v[ok_])))))
S3 = pd.DataFrame(rows)
print(S3.groupby(["bins", "src"]).apply(lambda d: pd.Series(dict(mean_r=d.r.mean(), sem_r=d.r.sem(), n_tests=len(d),
      n_p_lt_05_bins=int((d.p_bins < 0.05).sum()), n_p_lt_05_wsum=int((d.p_wsum < 0.05).sum()),
      min_p_bins=d.p_bins.min(), min_p_wsum=d.p_wsum.min()))).round(4).to_string())
print("expected average r 0.17 +- 0.02")
print(S3[(S3.bins == 10) & (S3.src == "rot")].round(4).to_string(index=False))
print(f"clean variant ({len(ENS_CR)} ensembles):")
print(S3[S3.session.isin(ENS_CR)].groupby(["bins", "src"]).apply(lambda d: pd.Series(dict(mean_r=d.r.mean(), n_tests=len(d),
      n_p_lt_05_bins=int((d.p_bins < 0.05).sum()), n_p_lt_05_wsum=int((d.p_wsum < 0.05).sum())))).round(4).to_string())


**Result: reproduced, resting on an assumption.** With 10 degree Idir bins and rotating-trial
probabilities, the average r is 0.166 (expected 0.17 ± 0.02), and none of the 14 weighted
Rayleigh tests has p < 0.05 with n the sum of the weights (smallest p 0.346). With n the number
of bins, one test has p = 0.0134 (1121R157, room frame). The other Idir readings give average r
0.180 to 0.264, with 3 or 4 of 14 tests below 0.05 when n is the number of bins and 1 to 3 when
n is the sum of the weights. The clean variant (12 tests) gives average r 0.154 under the main
reading, with no test below 0.05 when n is the sum of the weights.

## fig4-internal-tuning-preserved

> Internally referenced directional tuning of head-direction cell pairs (tuning of one cell to
> head-direction displacement since the other's spikes) was preserved across stable1, rotating
> and stable2; neither its instability nor its change in preference differed between trials or
> from the two halves of stable1.

For each pair of simultaneously recorded cells, cell 1 is the first of the pair in file order
(one ordering per pair; the ordered pairs, both ways, are a variant), and the histogram of the
displacement between the direction at each cell 1 spike and the direction at each cell 2 spike
within the interval (36 bins of 10 degrees). Main readings (criterion): 5 s interval, cell 2
spikes before the cell 1 spike (the code's side), the Methods' normalization by the sum of the
time offsets. Variants: 30 s (the code's window), cell 2 spikes after, the code's occupancy
normalization. Spike times are the times of the samples that hold them (index over rate), so
that spikes and samples share one clock. Conditions: S1, S2, RR (room directions), RA (arena
directions), and H1 and H2 (first and second 15 min of stable1; spikes and samples of the half).
Dividing by the sum of the time offsets scales each curve by one number, so under the main
reading the KLD (on counts), the preferred bin and the Rayleigh length are those of the raw
displacement histogram. That histogram also reflects how head direction moves over the interval,
not only the coupling of the two cells; the cell after the tests checks this with the tracker
samples alone. KLD as in fig1d on the histogram counts (the Methods' n_i; for the occupancy normalization on
the rate curve scaled to the histogram's total count); preference change as the circular
difference of the bins of maximum value. Pairs with an empty histogram in any condition are
dropped (criterion: "the pairs used are those with a defined curve in every trial"). Tests: a
repeated-measures ANOVA over the three trial pairs and a one-way ANOVA of the trial pairs and
the halves, each at p ≥ 0.05 (criterion).

In [ ]:
COND_IT = ("S1", "S2", "RR", "RA", "H1", "H2")
def events(key, tr, half=None):
    t = hc20.track(D, key, tr)
    i = spk[key][tr]["i"]
    n = len(t["d"])
    if half == 1:
        i = i[i < n // 2]
    elif half == 2:
        i = i[i >= n // 2]
    d = t["d"][i]
    ok_ = ~np.isnan(d)
    return i[ok_] / t["rate"], d[ok_]
def sample_events(s, tr, key, half=None):
    t = hc20.track(D, key, tr)
    n = len(t["d"])
    idx = np.arange(n)
    if half == 1:
        idx = idx[: n // 2]
    elif half == 2:
        idx = idx[n // 2:]
    d = t["d"][idx]
    ok_ = ~np.isnan(d)
    return hc20.bin_cum(idx[ok_] / t["rate"], d[ok_])
def cond_src(c):
    return {"S1": ("S1", None), "S2": ("S2", None), "RR": ("RR", None), "RA": ("RA", None), "H1": ("S1", 1), "H2": ("S1", 2)}[c]

def internal_all(pairs, intervals, side, norm):
    """Curves and histograms per pair, condition and interval."""
    out = {}
    for k1, k2 in pairs:
        for c in COND_IT:
            tr, half = cond_src(c)
            t1, d1 = events(k1, tr, half)
            t2, d2 = events(k2, tr, half)
            ev2 = hc20.bin_cum(t2, d2)
            evs = sample_events(k1[0], tr, k1, half) if norm == "occupancy" else None
            for L in intervals:
                out[(k1, k2, c, L)] = hc20.internal_tuning(t1, d1, ev2, evs, L, side=side, norm=norm)
    return out
def it_measures(IT, pairs, L, norm):
    rows = []
    for k1, k2 in pairs:
        cur = {c: IT[(k1, k2, c, L)] for c in COND_IT}
        if any(cur[c][1].sum() == 0 for c in COND_IT):
            continue
        def kt_input(c):
            curve, hist = cur[c]
            if norm == "sum_offsets":
                return hist
            return curve / curve.sum() * hist.sum() if curve.sum() > 0 else hist
        row = {"k1": k1, "k2": k2}
        for a_, b_ in (("S1", "S2"), ("S1", "RR"), ("S1", "RA"), ("H1", "H2")):
            row[f"kld_{a_}{b_}"] = hc20.kld(kt_input(a_), kt_input(b_))
            pa, pb = np.argmax(cur[a_][0]), np.argmax(cur[b_][0])
            row[f"pc_{a_}{b_}"] = hc20.circ_absdiff_deg(pa * 10, pb * 10)
        for c in COND_IT:
            row[f"str_{c}"] = hc20.resultant(cur[c][0], hc20.disp_centers())[0]
        row["clean"] = all((k, tr) not in EXCL for k in (k1, k2) for tr in ("RR", "RA", "S2"))
        rows.append(row)
    return pd.DataFrame(rows)
def fig4_tests(M, label):
    res = {}
    for m in ("kld", "pc"):
        cols = [f"{m}_S1S2", f"{m}_S1RR", f"{m}_S1RA"]
        a = hc20.rm_anova(M[cols].to_numpy())
        F, p = stats.f_oneway(*[M[c] for c in cols], M[f"{m}_H1H2"])
        res[m] = (a["p"], p)
        print(f"{label} [{m}] three trial pairs: {hc20.fmt_anova(a)}; one-way with halves: F(3, {4 * len(M) - 4}) = {F:.2f}, p = {p:.3g}; "
              f"means S1S2, S1RR, S1RA, halves = {np.round([M[c].mean() for c in cols + [f'{m}_H1H2']], 3)}")
    return res
t_ = time.time()
PAIRS_ORD = PAIRS + [(b_, a_) for a_, b_ in PAIRS]
IT_MAIN = internal_all(PAIRS, (5,), "before", "sum_offsets")
print(f"main readings computed in {time.time() - t_:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
M4 = it_measures(IT_MAIN, PAIRS, 5, "sum_offsets")
print(f"pairs: {len(PAIRS)}; with a curve in every condition: {len(M4)}")
R4 = {"main": fig4_tests(M4, "main (5 s, before, sum of offsets)")}
fig4_tests(M4[M4.clean], "main, clean variant")
for lab, (L, side, norm, prs) in {"30 s": (30, "before", "sum_offsets", PAIRS), "after": (5, "after", "sum_offsets", PAIRS),
                                   "occupancy": (5, "before", "occupancy", PAIRS), "ordered pairs": (5, "before", "sum_offsets", PAIRS_ORD)}.items():
    IT_v = internal_all(prs, (L,), side, norm)
    Mv = it_measures(IT_v, prs, L, norm)
    print(f"variant {lab}: pairs {len(Mv)}")
    R4[lab] = fig4_tests(Mv, f"variant {lab}")
print("strength (resultant length) of the 5 s curves, median by condition:", M4[[f"str_{c}" for c in COND_IT]].median().round(3).to_dict())

The instability is lower for stable1-arena than for stable1-room, though the two frames share
the same spikes and differ only by the rotation of the arena over the interval. The next cell
shows where the difference comes from: the spikes with a valid direction in each frame, the KLD
between the room- and arena-frame histograms, the circular mean displacement of each histogram,
and the same KLD from stable1 computed on the displacement histogram of the tracker samples alone
(head direction at each cell 1 spike against every sample in the window, scaled to the spike
histogram's total), which holds nothing of cell 2.

In [ ]:
# what drives the lower stable1-arena than stable1-room instability at 5 s (main readings):
# per pair, the spikes with a valid direction in each frame, the KLD between the room- and
# arena-frame histograms, the circular mean of each histogram, and the same KLD computed on the
# displacement histogram of the tracker samples alone (head direction at each cell 1 spike against
# every sample in the window), which holds no information from cell 2
def sample_hist(k1, c, L=5):
    tr, half = cond_src(c)
    t1, d1 = events(k1, tr, half)
    return hc20.internal_tuning(t1, d1, sample_events(k1[0], tr, k1, half), None, L, side="before", norm="none")[1]
rows = []
for k1, k2 in zip(M4.k1, M4.k2):
    r = {"k1": k1, "k2": k2}
    H = {c: IT_MAIN[(k1, k2, c, 5)][1] for c in ("S1", "S2", "RR", "RA")}
    for c in ("RR", "RA"):
        r[f"n1_{c}"], r[f"n2_{c}"] = len(events(k1, c)[0]), len(events(k2, c)[0])
    O = {c: sample_hist(k1, c) for c in ("S1", "S2", "RR", "RA")}
    for c in ("S1", "S2", "RR", "RA"):
        r[f"mean_disp_spikes_{c}"] = np.degrees(hc20.resultant(H[c], hc20.disp_centers())[1] + np.pi) % 360 - 180
        r[f"mean_disp_samples_{c}"] = np.degrees(hc20.resultant(O[c], hc20.disp_centers())[1] + np.pi) % 360 - 180
    r["kld_RR_RA"] = hc20.kld(H["RR"], H["RA"])
    sc = {c: O[c] / O[c].sum() * H[c].sum() for c in O}  # sample histograms scaled to the spike totals
    for b_ in ("S2", "RR", "RA"):
        r[f"kld_S1{b_}"] = hc20.kld(H["S1"], H[b_])
        r[f"kld_samples_S1{b_}"] = hc20.kld(sc["S1"], sc[b_])
    r["clean"] = all((k, tr) not in EXCL for k in (k1, k2) for tr in ("RR", "RA", "S2"))
    rows.append(r)
GAP = pd.DataFrame(rows)
print(f"pairs: {len(GAP)}; spikes with a valid direction differ between the room and arena frames in "
      f"{int(((GAP.n1_RR != GAP.n1_RA) | (GAP.n2_RR != GAP.n2_RA)).sum())} pairs; median cell 1 spikes room {GAP.n1_RR.median():.0f}, arena {GAP.n1_RA.median():.0f}; "
      f"median relative difference {np.median((GAP.n1_RR - GAP.n1_RA) / GAP.n1_RR):.3f}")
print("KLD between the room- and arena-frame histograms of a pair: median", round(GAP.kld_RR_RA.median(), 3), "; mean", round(GAP.kld_RR_RA.mean(), 3))
print("median circular mean displacement (deg), spike histograms:", GAP[[f"mean_disp_spikes_{c}" for c in ("S1", "S2", "RR", "RA")]].median().round(1).to_dict())
print("median circular mean displacement (deg), tracker samples only:", GAP[[f"mean_disp_samples_{c}" for c in ("S1", "S2", "RR", "RA")]].median().round(1).to_dict())
for lab, g in (("all pairs", GAP), ("clean pairs", GAP[GAP.clean])):
    t_sp = stats.ttest_rel(g.kld_S1RR, g.kld_S1RA)
    t_sa = stats.ttest_rel(g.kld_samples_S1RR, g.kld_samples_S1RA)
    print(f"{lab} (n = {len(g)}): mean KLD from stable1, spikes: S2 {g.kld_S1S2.mean():.3f}, room {g.kld_S1RR.mean():.3f}, arena {g.kld_S1RA.mean():.3f} "
          f"(room - arena paired t p = {t_sp.pvalue:.3g}); tracker samples only: S2 {g.kld_samples_S1S2.mean():.3f}, room {g.kld_samples_S1RR.mean():.3f}, "
          f"arena {g.kld_samples_S1RA.mean():.3f} (paired t p = {t_sa.pvalue:.3g})")
print("per pair, r between (room - arena) KLD from spikes and from samples only:",
      round(stats.pearsonr(GAP.kld_S1RR - GAP.kld_S1RA, GAP.kld_samples_S1RR - GAP.kld_samples_S1RA)[0], 3))


**Result: not reproduced.** Under the main readings (5 s, cell 2 spikes before, sum of
offsets; 142 pairs) the instability differs across trial pairs (p = 8.27e-05) and between the
trial pairs and the halves (p = 0.026); the preference change does not (p = 0.0528 and
p = 0.303). The instability is lowest for stable1-arena (0.156 against 0.196, 0.214 and 0.187).
The room- and arena-frame histograms differ by a shift of the displacement: the median circular
mean is -4.6 degrees in the room frame and 6.3 in the arena frame, against 4.0 in stable1, and the
tracker samples alone show the same shift (-4.8, 6.9, 3.3) and the same order of KLD from stable1
(room 0.068, arena 0.048, paired p = 7.55e-08). The valid spike counts of the two frames differ by
a median of 1.4%. So the room-arena gap that drives the significant test follows the passive
rotation of head direction in the room frame, a property of the measure; per pair the two gaps
are not correlated (r = -0.01). In the clean variant all four tests have p ≥ 0.05 (0.0991, 0.228,
0.227, 0.744). The 30 s, after-side, occupancy and ordered-pair variants each have at least one
test with p < 0.05. The claim is one of absence, judged on the paper's test as deposited, so the
outcome is not reproduced.

## fig4c-arena-registration-short-lived

> The strength of internally referenced directional tuning is similar in all conditions and
> frames for intervals up to 10 s, but beyond 10 s it falls in the rotating arena frame (20 to
> 60 s; stable1 = stable2 = rotating room > rotating arena) and at 300 to 600 s also in the
> rotating room frame (stable1 = stable2 > rotating room > rotating arena).

Strength is read as the Rayleigh vector length of the internally referenced curve (criterion),
per pair at maximum intervals of 3, 4, 5, 10, 20, 30, 40, 50, 60, 300 and 600 s, in S1, S2, RR
and RA; a one-way ANOVA of the four conditions at each interval with Tukey-Kramer HSD
(criterion), under the main readings of fig4 and with the occupancy normalization, the
after side and the clean pairs (without the unit-trials shown wrong) as variants.

In [ ]:
INTS = (3, 4, 5, 10, 20, 30, 40, 50, 60, 300, 600)
C4 = ("S1", "S2", "RR", "RA")
def fig4c(side, norm, label, clean_only=False):
    t_ = time.time()
    IT = internal_all(PAIRS, INTS, side, norm)
    rows = []
    for L in INTS:
        M = it_measures(IT, PAIRS, L, norm)
        if clean_only:
            M = M[M.clean]
        groups = {c: M[f"str_{c}"].to_numpy() for c in C4}
        F, p = stats.f_oneway(*groups.values())
        tk = hc20.tukey(groups)
        def pt(a_, b_):
            r = tk[((tk.group1 == a_) & (tk.group2 == b_)) | ((tk.group1 == b_) & (tk.group2 == a_))].iloc[0]
            return float(r["p-adj"])
        rows.append(dict(interval=L, n_pairs=len(M), **{f"mean_{c}": groups[c].mean() for c in C4}, F=F, p=p,
                         p_RA_S1=pt("RA", "S1"), p_RA_S2=pt("RA", "S2"), p_RA_RR=pt("RA", "RR"), p_RR_S1=pt("RR", "S1"),
                         p_RR_S2=pt("RR", "S2"), p_S1_S2=pt("S1", "S2")))
    df = pd.DataFrame(rows)
    print(f"{label} (computed in {time.time() - t_:.0f} s):")
    print(df.round(4).to_string(index=False))
    short, longi = df[df.interval <= 10], df[df.interval > 10]
    arena_low = all((r.mean_RA < min(r.mean_S1, r.mean_S2, r.mean_RR)) and max(r.p_RA_S1, r.p_RA_S2, r.p_RA_RR) < 0.05
                    for r in df[(df.interval >= 20) & (df.interval <= 60)].itertuples())
    room_low = all((r.mean_RR < min(r.mean_S1, r.mean_S2)) and max(r.p_RR_S1, r.p_RR_S2) < 0.05 for r in df[df.interval >= 300].itertuples())
    print(f"  ANOVA p >= 0.05 at every interval up to 10 s: {bool((short.p >= 0.05).all())}; p < 0.05 at every interval beyond 10 s: "
          f"{bool((longi.p < 0.05).all())}; arena frame below the other three (Tukey p < 0.05) at 20 to 60 s: {arena_low}; "
          f"room frame below both stable trials at 300 to 600 s: {room_low}")
    return df
F4C = {"main": fig4c("before", "sum_offsets", "main (before, sum of offsets)")}
F4C["occupancy"] = fig4c("before", "occupancy", "variant, occupancy normalization")
F4C["after"] = fig4c("after", "sum_offsets", "variant, cell 2 spikes after")
F4C["clean"] = fig4c("before", "sum_offsets", "clean variant, main readings, pairs with clean files", clean_only=True)
print(f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

**Result: not reproduced.** Under the main readings the conditions do not differ at 3, 4 and
5 s (p = 0.1747, 0.2438, 0.1703) but do at 10 s (p = 0.0106), and differ at every interval beyond
10 s (p < 0.0001). At 20 to 60 s the arena frame is not below the room frame: the room frame
falls first and is below the arena frame at 20 to 40 s (means 0.378 against 0.438 at 20 s), with
room against arena Tukey p = 0.0611 to 0.9176. At 300 and 600 s the room frame is below both
stable trials and the arena frame below the room frame (p = 0.0135 and 0.0046). The criterion's
partial outcome needs the arena frame to fall below the others beyond 10 s and not before, which
fails at 10 s and at 20 to 60 s, so the claim is not reproduced. The occupancy and after-side
variants give the same pattern at 20 to 60 s, and the clean variant (112 to 118 pairs) differs
already at 3 to 5 s (p = 0.0109, 0.0198, 0.0127) and puts the arena frame below the room frame
at p < 0.05 only from 60 s (p = 0.0368).

## Blocked claims

- fig1a-avoidance-learning: the learning curves come from the pretraining and training sessions;
  the dataset holds only the stable1, rotating, stable2 recording trials (the .dat files under
  data/DAT/SA and data/DAT/RA are those trials, 88 files each for 44 sessions).
- fig1c-lfp-power: no LFP is in the dataset.
- figs4-all-pair-correlations-preserved: needs the 353 MEC cells of Table S1; the dataset holds
  only the 115 head-direction cells.
- figs5-hd-cells-high-model-information and figs6-subclass-correlations-preserved: need the
  921 modelled MEC cells; only the head-direction cells are in the dataset.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| cell-counts | partially reproduced (115 head-direction cells from 13 rats; MEC counts not in the dataset) |
| hd-cell-rate | reproduced |
| fig1a-avoidance-learning | blocked: data missing |
| fig1c-speed | reproduced, resting on an assumption |
| fig1c-lfp-power | blocked: data missing |
| fig1c-firing-rate | reproduced |
| fig1c-directionality | reproduced, resting on an assumption |
| fig1d-preference-change | partially reproduced |
| fig1d-instability | partially reproduced |
| fig2-decoding-stable-vs-rotating | reproduced, resting on an assumption |
| figs2-frame-preference | reproduced, resting on an assumption |
| fig2c-proximity-anticorrelation | partially reproduced |
| figs3-no-head-direction-bias | reproduced, resting on an assumption |
| fig3-hd-pair-correlations-preserved | reproduced |
| figs4-all-pair-correlations-preserved | blocked: data missing |
| figs5-hd-cells-high-model-information | blocked: data missing |
| figs6-subclass-correlations-preserved | blocked: data missing |
| no-angular-velocity-tuning | not reproduced, resting on an assumption |
| fig4-internal-tuning-preserved | not reproduced |
| fig4c-arena-registration-short-lived | not reproduced |
| fig5-error-inhomogeneous | partially reproduced |
| fig5c-error-minima-at-cues | reproduced |

**How the analysis differs from the paper's.** The outcomes are decided on the files as
deposited, all 115 cells; a clean variant leaves out the unit-trials whose files are shown wrong
in the data check. Four outcomes rest on files shown wrong: fig2 and figs2 meet their counts
only with 1023R029, and fig1c-directionality and fig4 change outcome in the clean variant. The
post hoc test after a repeated-measures ANOVA uses the ANOVA's error term, an assumption, with
the between-groups Tukey test as a variant; fig1c-directionality depends on it. Several details the paper leaves open (Idir
bins and probabilities, the 2 SD rule, relative proximity, the n of the Rayleigh test of Figure
S3, the normalization of internally referenced tuning, how speed is computed) are run under
each reading the claims file names, as shown in each section. Spike width is not tested.

**What was found about the dataset.**

- It holds only the 115 head-direction cells; the other MEC cells, the LFP, the waveforms and
  the training sessions are not in it. Per rat, the counts match Table S1 except R029 (40
  against 41) and R523 (2 against 1). The 44 sessions give 146 simultaneously recorded pairs,
  against 142 in the paper.
- FileNameList.xlsx misaligns the stable2 column for five units of session 1121R157.
- The rotating spike file of unit 1024R157 P9C1 holds its stable1 spike times.
- The rotating room-frame track of session 1211R039 is that of session 1212R039.
- In sessions 1009R029, 1015R029, 1023R029 and 1025R029, the room-frame and arena-frame tracks
  of the rotating trial disagree with each other and with the tracker files, with a lag that
  grows through the trial.
- The stable2 file of 1121R157 P11C2 holds a track of half the trial; the stable2 header of
  1017R035 P11C2 states 2000 samples/s.
- Spike timestamps run about 0.05% faster than the tracker sample count at the header's rate.